# Intention Collapse - Experimento Piloto en Colab

**GPU:** A100  
**Objetivo:** Correr experimento pequeño (50-100 problemas) para validar pipeline

## 1. Setup: Clonar repo e instalar dependencias

In [ ]:
# Verificar GPU
!nvidia-smi

In [ ]:
# Clonar repo (si no está clonado)
import os
if not os.path.exists('intention-collapse-experiments'):
    !git clone https://github.com/patriciomvera/intention-collapse-experiments.git
    
%cd intention-collapse-experiments

In [ ]:
# Instalar dependencias
!pip install -q torch transformers datasets accelerate bitsandbytes scikit-learn pandas numpy matplotlib seaborn tqdm pyyaml

## 2. Montar Google Drive (para guardar outputs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
OUTPUT_BASE = Path('/content/drive/MyDrive/intention_collapse_outputs')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print(f"✓ Outputs se guardarán en: {OUTPUT_BASE}")

## 3. Imports y configuración

In [ ]:
import sys
sys.path.append(str(Path.cwd()))

# Importar módulos (ajusta según tengas shared_utils o módulos separados)
try:
    from src import shared_utils as U
    print("✓ Usando shared_utils.py (v3)")
except ImportError:
    print("⚠️ shared_utils.py no encontrado, importando módulos legacy")
    from src import activation_hooks, data_utils, metrics, probing

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import json
from tqdm.auto import tqdm

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 4. Cargar modelo y dataset

In [ ]:
# Configuración
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
SAMPLE_SIZE = 50  # Piloto pequeño
LAYERS = [27, 28, 29, 30, 31]  # Últimas capas

# Cargar modelo (4-bit)
print("Cargando modelo...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Modelo cargado: {MODEL_NAME}")

In [ ]:
# Cargar GSM8K
print("Cargando GSM8K...")
dataset = load_dataset("openai/gsm8k", "main", split="test")
sample_dataset = dataset.select(range(SAMPLE_SIZE))

print(f"✓ Dataset cargado: {SAMPLE_SIZE} problemas")

## 5. Definir prompts de los regímenes

In [ ]:
PROMPTS = {
    "baseline": "Solve this math problem. Give only the final numerical answer.\n\nProblem: {question}\n\nAnswer:",
    "cot": "Solve this math problem step by step. Show your reasoning, then give the final answer after ####.\n\nProblem: {question}\n\nSolution:",
    "babble": "Given this math problem, write a long stream of consciousness about numbers and calculations. Do NOT solve it.\n\nProblem: {question}\n\nStream:"
}

MAX_TOKENS = {
    "baseline": 50,
    "cot": 512,
    "babble": 512
}

## 6. Función para extraer activaciones

In [ ]:
def extract_activations(model, tokenizer, prompt, layers, max_tokens=50):
    """Extrae activaciones de las capas especificadas."""
    activations = {layer: [] for layer in layers}
    
    def hook_fn(layer_idx):
        def hook(module, input, output):
            # output[0] es el tensor de activaciones
            activations[layer_idx].append(output[0].detach().cpu())
        return hook
    
    # Registrar hooks
    hooks = []
    for layer_idx in layers:
        layer = model.model.layers[layer_idx]
        hook = layer.register_forward_hook(hook_fn(layer_idx))
        hooks.append(hook)
    
    # Generar
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,  # Greedy
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Limpiar hooks
    for hook in hooks:
        hook.remove()
    
    # Decodificar output
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return activations, generated_text

## 7. Correr experimento piloto

In [ ]:
results = []

for idx, item in enumerate(tqdm(sample_dataset, desc="Procesando problemas")):
    question = item['question']
    ground_truth = item['answer'].split('####')[-1].strip()
    
    item_results = {
        'idx': idx,
        'question': question,
        'ground_truth': ground_truth,
        'regimes': {}
    }
    
    # Probar cada régimen
    for regime in ['baseline', 'cot']:  # Omitimos babble para ir más rápido
        prompt = PROMPTS[regime].format(question=question)
        max_tok = MAX_TOKENS[regime]
        
        try:
            activations, output = extract_activations(model, tokenizer, prompt, LAYERS, max_tok)
            
            item_results['regimes'][regime] = {
                'output': output,
                'activations_shapes': {k: [a.shape for a in v] for k, v in activations.items()}
            }
        except Exception as e:
            print(f"Error en {idx}/{regime}: {e}")
            item_results['regimes'][regime] = {'error': str(e)}
    
    results.append(item_results)
    
    # Checkpoint cada 10 problemas
    if (idx + 1) % 10 == 0:
        checkpoint_path = OUTPUT_BASE / f'checkpoint_{idx+1}.json'
        with open(checkpoint_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"✓ Checkpoint guardado: {checkpoint_path}")

print("\n✅ Experimento completado!")

## 8. Guardar resultados finales

In [ ]:
final_path = OUTPUT_BASE / 'pilot_results.json'
with open(final_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Resultados guardados en: {final_path}")
print(f"Problemas procesados: {len(results)}")

## 9. Preview de resultados

In [ ]:
# Ver primer resultado
print("Ejemplo de resultado:")
print(json.dumps(results[0], indent=2))